# 🖼️ Image Caption API — Google Colab

Serve LLaVA JoyCaption as a simple HTTP API from Colab. Accepts multiple images per request via multipart form-data, JSON base64, or a ZIP of frames.

## Quick Start
1. Runtime → Change runtime type → GPU (T4 recommended)
2. Run all cells in order
3. Use the printed local URL, or the Colab proxy URL (for your session)

## Endpoints
- GET /health — model/device info
- POST /caption — multipart images (repeat `images` fields)
- POST /caption-json — JSON base64 payload
- POST /caption-zip — ZIP with image files

Note: Colab proxy URLs work only in your session. They are not public.


In [ ]:
# Check GPU
!nvidia-smi || echo 'No NVIDIA GPU visible'

import os, sys, platform, time
print(f'Python: {sys.version.split()[0]}  |  Platform: {platform.platform()}')


In [ ]:
# Install dependencies
!pip -q install 
  'transformers>=4.44.0' accelerate bitsandbytes pillow 
  fastapi uvicorn python-multipart pydantic 

print('✅ Dependencies installed')


In [ ]:
# Environment configuration (GPU + Transformers)
import os, torch
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'

if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {DEVICE}')


In [ ]:
# Model loading
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image

MODEL_NAME = 'fancyfeast/llama-joycaption-alpha-two-hf-llava'
print(f'📥 Loading model: {MODEL_NAME}')

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE=='cuda' else torch.float32,
    device_map='auto',
    attn_implementation='eager',
    trust_remote_code=True,
)
model.eval()
print(f'✅ Model ready on {DEVICE}')


In [ ]:
# Helper functions (downscale, dtype/device normalization, captioning)
import io, base64, re, asyncio, json, tempfile, zipfile
from typing import List, Optional, Tuple

MAX_SIDE_DEFAULT = 672

def downscale_image(img: Image.Image, max_side: int = MAX_SIDE_DEFAULT) -> Image.Image:
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    scale = max_side / float(max(w, h))
    return img.resize((int(w*scale), int(h*scale)), Image.BICUBIC)

def _normalize_inputs_for_generate(inputs, device):
    fixed = {}
    for k, v in inputs.items():
        if not hasattr(v, 'to'):
            fixed[k] = v
            continue
        if k == 'pixel_values':
            if v.dtype != (torch.float16 if device=='cuda' else torch.float32):
                v = v.to(torch.float16 if device=='cuda' else torch.float32)
        elif k == 'input_ids':
            if v.dtype != torch.int64:
                v = v.to(torch.int64)
        elif k in ('attention_mask','pixel_attention_mask','cross_attention_mask'):
            if v.dtype != torch.int64:
                v = v.to(torch.int64)
        v = v.to(device)
        fixed[k] = v
    return fixed

def caption_single_image(img: Image.Image, prompt: Optional[str] = None, *, max_new_tokens=96, temperature=0.6, top_p=0.9) -> str:
    if prompt is None:
        prompt = 'Write a concise, descriptive caption for this image.'
    convo = [
        {'role': 'system', 'content': 'You are a concise, visual captioner.'},
        {'role': 'user',   'content': prompt},
    ]
    tmpl = processor.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)
    raw_inputs = processor(text=[tmpl], images=[img], return_tensors='pt', padding=True)
    inputs = _normalize_inputs_for_generate(raw_inputs, DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=top_p, use_cache=True
        )[0]
    out = out[inputs['input_ids'].shape[1]:]
    text = processor.tokenizer.decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return text

def caption_batch(images: List[Image.Image], *, prompt: Optional[str] = None, max_new_tokens=96, temperature=0.6, top_p=0.9) -> List[str]:
    caps = []
    for img in images:
        cap = caption_single_image(img, prompt, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p)
        caps.append(cap)
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
    return caps

DATA_URL_RE = re.compile(r'^data:.*?;base64,(.*)$')
def decode_base64_image(data: str) -> Image.Image:
    m = DATA_URL_RE.match(data)
    if m:
        data = m.group(1)
    b = base64.b64decode(data)
    return Image.open(io.BytesIO(b)).convert('RGB')

INFERENCE_LOCK = asyncio.Lock()
MAX_IMAGES_PER_REQUEST = int(os.environ.get('MAX_IMAGES_PER_REQUEST', '32'))


In [ ]:
# FastAPI app and endpoints
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

app = FastAPI(title='Image Caption API', version='1.0.0')

@app.get('/health')
def health():
    return {
        'status': 'ok',
        'device': DEVICE,
        'model': MODEL_NAME,
        'cuda': torch.cuda.is_available(),
    }

@app.post('/caption')
async def caption_endpoint(
    images: List[UploadFile] = File(..., description='Repeat this field for multiple images'),
    timestamps: Optional[str] = Form(None, description='Optional JSON array or comma-separated times'),
    prompt: Optional[str] = Form(None),
    max_new_tokens: int = Form(96),
    temperature: float = Form(0.6),
    top_p: float = Form(0.9),
    max_side: int = Form(MAX_SIDE_DEFAULT),
):
    if not images:
        raise HTTPException(400, 'No images provided')
    if len(images) > MAX_IMAGES_PER_REQUEST:
        raise HTTPException(400, f'Max {MAX_IMAGES_PER_REQUEST} images per request')

    ts_list: Optional[List[str]] = None
    if timestamps:
        try:
            ts_list = json.loads(timestamps) if timestamps.strip().startswith('[') else [s.strip() for s in timestamps.split(',')]
        except Exception:
            raise HTTPException(400, 'Invalid timestamps format')

    start = time.time()
    async with INFERENCE_LOCK:
        pil_images: List[Image.Image] = []
        for f in images:
            b = await f.read()
            try:
                img = Image.open(io.BytesIO(b)).convert('RGB')
            except Exception:
                raise HTTPException(400, f'Invalid image: {f.filename}')
            pil_images.append(downscale_image(img, max_side=max_side))
        caps = caption_batch(pil_images, prompt=prompt, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p)

    results = []
    for i, cap in enumerate(caps):
        item = {'index': i, 'caption': cap}
        if ts_list and i < len(ts_list):
            item['timestamp'] = ts_list[i]
        results.append(item)

    return JSONResponse({
        'results': results,
        'metadata': {
            'count': len(results),
            'processing_time_seconds': round(time.time()-start, 3),
            'max_side': max_side,
            'device': DEVICE,
        }
    })

class CaptionJSONRequest(BaseModel):
    images: List[str]
    timestamps: Optional[List[str]] = None
    prompt: Optional[str] = None
    max_new_tokens: int = 96
    temperature: float = 0.6
    top_p: float = 0.9
    max_side: int = MAX_SIDE_DEFAULT

@app.post('/caption-json')
async def caption_json(req: CaptionJSONRequest):

    imgs_b64 = req.images or []
    if not imgs_b64:
        raise HTTPException(400, 'No images provided')
    if len(imgs_b64) > MAX_IMAGES_PER_REQUEST:
        raise HTTPException(400, f'Max {MAX_IMAGES_PER_REQUEST} images per request')

    start = time.time()
    async with INFERENCE_LOCK:
        pil_images = []
        for s in imgs_b64:
            try:
                img = decode_base64_image(s)
            except Exception:
                raise HTTPException(400, 'Invalid base64 image')
            pil_images.append(downscale_image(img, max_side=req.max_side))
        caps = caption_batch(pil_images, prompt=req.prompt, max_new_tokens=req.max_new_tokens, temperature=req.temperature, top_p=req.top_p)

    results = []
    for i, cap in enumerate(caps):
        item = {'index': i, 'caption': cap}
        if req.timestamps and i < len(req.timestamps):
            item['timestamp'] = req.timestamps[i]
        results.append(item)

    return {
        'results': results,
        'metadata': {
            'count': len(results),
            'processing_time_seconds': round(time.time()-start, 3),
            'max_side': req.max_side,
            'device': DEVICE,
        }
    }

@app.post('/caption-zip')
async def caption_zip(frames_zip: UploadFile = File(...), prompt: Optional[str] = Form(None), max_new_tokens: int = Form(96), temperature: float = Form(0.6), top_p: float = Form(0.9), max_side: int = Form(MAX_SIDE_DEFAULT)):

    data = await frames_zip.read()
    start = time.time()
    async with INFERENCE_LOCK:
        with tempfile.TemporaryDirectory() as td:
            zp = zipfile.ZipFile(io.BytesIO(data))
            zp.extractall(td)
            names = sorted([n for n in zp.namelist() if n.lower().endswith(('.jpg','.jpeg','.png','.webp','.bmp'))])
            if not names:
                raise HTTPException(400, 'ZIP contains no images')
            if len(names) > MAX_IMAGES_PER_REQUEST:
                raise HTTPException(400, f'Max {MAX_IMAGES_PER_REQUEST} images per request')
            pil_images = []
            for n in names:
                p = os.path.join(td, n)
                img = Image.open(p).convert('RGB')
                pil_images.append(downscale_image(img, max_side=max_side))
            caps = caption_batch(pil_images, prompt=prompt, max_new_tokens=max_new_tokens, temperature=temperature, top_p=top_p)

    results = [{'index': i, 'filename': names[i], 'caption': c} for i, c in enumerate(caps)]
    return {
        'results': results,
        'metadata': {
            'count': len(results),
            'processing_time_seconds': round(time.time()-start, 3),
            'max_side': max_side,
            'device': DEVICE,
        }
    }


In [ ]:
# Launch server (local)
import threading, uvicorn

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, workers=1)

t = threading.Thread(target=run_server, daemon=True)
t.start()
print('✅ Uvicorn launched on http://127.0.0.1:8000')
print('   Use /health to check status')


### Colab Proxy URL
Use Colab's built-in proxy to access the API from your browser session. The printed URL maps to `http://127.0.0.1:8000/`.
Note: This proxy works only while the notebook is running and only for your session; it is not publicly accessible.


In [ ]:
# Print Colab proxy URL for this session
try:
    from google.colab import output as colab_output
    proxy_base = colab_output.eval_js('google.colab.kernel.proxyPort(8000)')
    if proxy_base and isinstance(proxy_base, str):
        if not proxy_base.endswith('/'):
            proxy_base += '/'
        print('🔗 Colab proxy base:', proxy_base)
        print('   Health:', proxy_base + 'health')
        print('   Caption:', proxy_base + 'caption')
    else:
        print('Colab proxy URL unavailable.')
except Exception as e:
    print('Colab proxy not available or not in Colab:', e)


(Removed) Tunneling via ngrok/cloudflared — using Colab proxy instead.


## Testing
Curl (multipart, inside Colab session via proxy):
```bash
# Replace <PROXY_BASE> with the printed proxy_base value
# curl example:
curl -X POST '<PROXY_BASE>caption' \
  -F 'images=@/path/img0.jpg' -F 'images=@/path/img1.jpg'
```

JSON base64 (Python):
```python
import base64, requests
from google.colab import output as colab_output
proxy_base = colab_output.eval_js('google.colab.kernel.proxyPort(8000)')
if not proxy_base.endswith('/'):
    proxy_base += '/'
def b64(path):
    return 'data:image/jpeg;base64,' + base64.b64encode(open(path,'rb').read()).decode()
payload = {'images':[b64('img0.jpg'), b64('img1.jpg')]}
r = requests.post(proxy_base + 'caption-json', json=payload)
print(r.json())
```
